# Kokoro-Hindi teacher cache (one run, everything training needs)

This notebook turns the 12,000-text Hindi corpus into a **reusable teacher cache**: every
row stores the Kokoro waveform **and** the intermediates (`t_en`, `d`, `F0`, `N`, `dur`,
`pred_dur`, `ids`, `style`), so no future student ever has to run Kokoro again.

**Before you run**

1. Accelerator: **GPU T4 x2** (one T4 also works, just slower).
2. Internet: **on** (GitHub, Hugging Face, pip).
3. Attach your `selected_texts.jsonl` as a Kaggle Dataset if you have it. If it is not
   attached, the notebook downloads the public `subhamgiri9748/kokoros-distill` corpus.
4. Optional publishing: add **Kaggle secrets** `KAGGLE_USERNAME` and `KAGGLE_KEY` (Create
   → Add-ons → Secrets), or attach a `kaggle.json`. Without them the cache is still built
   and the notebook prints the manual upload command.
5. Run **interactively**, not "Save & Run All": the cache is 15–16 GB and Kaggle notebook
   output is smaller than that. Publish with the upload cell before the session ends.

Expected: ~2–4 h for 13,655 segments on 2×T4, then ~5–10 min validation and ~30–60 min
upload. The generation is resumable: if the session dies, rerun the generation cell and
completed shards are skipped.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

REPO_URL = "https://github.com/Subham460/hinditts.git"
REPO_REF = "main"
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "hinditts"
TRAIN_DIR = REPO_DIR / "training"
BUILD_DIR = WORK / "teacher_build"
SMOKE_DIR = WORK / "teacher_smoke"

SHARD_SIZE = 500
MAX_PHONEMES = 400
SEED = 0
N_PROCS = min(2, max(1, torch.cuda.device_count()))

# "kaggle", "hf", or "none".  "kaggle" needs KAGGLE_USERNAME/KAGGLE_KEY.
PUBLISH = "kaggle"
KAGGLE_SLUG = "kokoro-hindi-teacher"
HF_REPO_ID = ""          # only used when PUBLISH == "hf"

# ~16 GB of shards.  /kaggle/working is resumable; fall back to /kaggle/temp if needed.
need_gb = 17.0
DATASET_DIR = TRAIN_DIR / "data" / "teacher"
work_free = shutil.disk_usage(WORK).free / 1e9
temp = Path("/kaggle/temp")
if work_free < need_gb and temp.exists() and shutil.disk_usage(temp).free / 1e9 >= need_gb:
    DATASET_DIR = temp / "teacher"
    print(f"working disk has only {work_free:.1f} GB free; writing to {DATASET_DIR}")

print("python   ", sys.version.split()[0])
print("torch    ", torch.__version__, "cuda", torch.version.cuda)
print("gpus     ", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory / 1e9:.1f} GB")
print("workspace", WORK)
print("dataset  ", DATASET_DIR)
u = shutil.disk_usage(WORK)
print(f"disk      {u.used / 1e9:.1f} used / {u.total / 1e9:.1f} GB, {u.free / 1e9:.1f} GB free")
assert torch.cuda.is_available(), "Set Accelerator to 'GPU T4 x2'"

In [ ]:
# Clone the public repo at the current main, then install only what generation needs.
# Do not `uv sync`: Kaggle already has torch, and a second environment wastes ~5 GB.
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_REF}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                        capture_output=True, text=True).stdout.strip()
print("repo:", commit)
print("training dir:", TRAIN_DIR)
assert (TRAIN_DIR / "scripts" / "gen_teacher_dataset.py").is_file()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                "kokoro==0.9.4", "misaki[en]==0.9.4", "huggingface_hub>=0.27"], check=True)

import importlib.metadata as md
for name in ("kokoro", "misaki", "transformers", "huggingface-hub", "torch"):
    try:
        print(f"{name:18s}", md.version(name))
    except Exception:
        print(f"{name:18s}", "not installed")
from kokoro import KPipeline
print("kokoro import ok")

## 1. Find or download the source records

The preferred input is the 12,000-row selection with stable `id` and `text` fields
(`selected_texts.jsonl`). If it is not attached, the public dataset is downloaded. The
download also writes an id-preserving JSONL, so either path keeps source ids.

In [ ]:
def find_source():
    patterns = [
        "/kaggle/input/*/selected_texts.jsonl",
        "/kaggle/input/*/*/selected_texts.jsonl",
        "/kaggle/input/*/hindi_sents.jsonl",
        "/kaggle/input/*/*/hindi_sents.jsonl",
        "/kaggle/input/*/hindi_sents.json",
        "/kaggle/input/*/*/hindi_sents.json",
    ]
    for pattern in patterns:
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    p = WORK / "kokoro_distill_selection" / "selected_texts.jsonl"
    return str(p) if p.exists() else None

os.chdir(TRAIN_DIR)
SOURCE = find_source()
if SOURCE is None:
    print("No source attached; downloading the public 12k corpus (internet required)...")
    subprocess.run([sys.executable, "scripts/fetch_hindi_sents.py", "--n", "12000", "--workers", "1"],
                   cwd=TRAIN_DIR, check=True)
    SOURCE = "data/hindi_sents.jsonl"
print("source:", SOURCE)

src = Path(SOURCE)
if src.suffix == ".jsonl":
    n = 0
    ids = set()
    duplicate_ids = 0
    with open(src, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            n += 1
            assert isinstance(row, dict) and str(row.get("text", "")).strip(), f"bad record at line {n}"
            if row.get("id") is not None:
                key = str(row["id"])
                duplicate_ids += key in ids
                ids.add(key)
    print(f"records {n:,}  with ids {len(ids):,}  duplicate ids {duplicate_ids}")
    assert n >= 1000, "source looks truncated"
    assert duplicate_ids == 0, "source ids are not unique"
    if n < 12000:
        print(f"WARNING: only {n:,} source rows; the usual corpus has 12,000")
else:
    data = json.load(open(src, encoding="utf-8"))
    if isinstance(data, dict):
        data = next(data[k] for k in ("rows", "data", "records") if isinstance(data.get(k), list))
    print("records", len(data))
    assert len(data) >= 1000

## 2. Build the segment manifest

Long rows are split at word boundaries into ≤400-phoneme segments, keeping the parent id
and a stable parent-level train/val/test split. Duplicate segment *text* is kept (with a
unique id) so no source row loses a segment; `duplicate_count` reports it.

In [ ]:
BUILD_DIR.mkdir(parents=True, exist_ok=True)
manifest = "data/hindi_segments.jsonl"
report_path = "data/hindi_segments_report.json"

cmd = [sys.executable, "scripts/prepare_hindi.py",
       "--in", str(src),
       "--out", manifest,
       "--texts-out", str(BUILD_DIR / "hindi_sents_segments.json"),
       "--report", report_path,
       "--max", str(MAX_PHONEMES),
       "--seed", str(SEED)]
print(" ".join(cmd))
subprocess.run(cmd, cwd=TRAIN_DIR, check=True)

report = json.load(open(TRAIN_DIR / report_path, encoding="utf-8"))
for key in ("records", "records_with_ids", "segments", "unique_segments",
            "duplicate_count", "counts", "phonemes"):
    print(f"{key:18s}", json.dumps(report[key], ensure_ascii=False))

assert report["segments"] == report["counts"]["total"]
assert report["counts"]["total"] == sum(report["counts"][k] for k in ("train", "val", "test"))
assert report["phonemes"]["over_max"] == 0, "a segment is over the limit"
assert report["segments"] >= report["records"], "segmentation lost source rows"
assert report["counts"]["val"] > 0 and report["counts"]["test"] > 0
print("ok: every source row is represented and no segment is over the limit")

In [ ]:
# Independent preflight: re-read the manifest and check the invariants training relies on.
import hashlib
from collections import Counter

rows = [json.loads(line) for line in open(TRAIN_DIR / manifest, encoding="utf-8") if line.strip()]
ids = [r["id"] for r in rows]
parents = {r["parent_id"] for r in rows}
assert len(ids) == len(set(ids)), "duplicate segment id"
assert len(parents) == report["records"], "not every source row has a segment"
assert all(r["phonemes"] and r["phoneme_count"] == len(r["phonemes"]) <= MAX_PHONEMES for r in rows)
assert all(r["split"] in ("train", "val", "test") for r in rows)
assert all(r["text_hash"] == hashlib.sha256(r["text"].encode("utf-8")).hexdigest() for r in rows)

by_parent = {}
for r in rows:
    by_parent.setdefault(r["parent_id"], set()).add(r["split"])
assert all(len(v) == 1 for v in by_parent.values()), "a parent row spans splits"

lens = sorted(r["phoneme_count"] for r in rows)
n_shards = (len(rows) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"segments {len(rows):,}  parents {len(parents):,}  shards {n_shards} x {SHARD_SIZE}")
print("splits  ", dict(Counter(r["split"] for r in rows)))
print(f"phonemes min {lens[0]}  p50 {lens[len(lens)//2]}  p95 {lens[int(len(lens)*.95)]}  max {lens[-1]}")
print(f"planned per GPU: {len(range(0, n_shards, N_PROCS))} / {len(range(1, n_shards, N_PROCS))} shards")

## 3. Smoke test on 4 rows

This verifies the teacher revision, GPU path, row schema, and validator before spending
hours. The smoke shards go to a separate directory and are deleted afterwards.

In [ ]:
TEACHER_REVISION = ""
try:
    from huggingface_hub import HfApi
    TEACHER_REVISION = HfApi().model_info("hexgrad/Kokoro-82M").sha or ""
except Exception as e:
    print("WARNING: could not resolve the Kokoro revision:", e)
print("teacher revision:", TEACHER_REVISION or "(each process will resolve HEAD)")

shutil.rmtree(SMOKE_DIR, ignore_errors=True)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONUNBUFFERED"] = "1"
cmd = [sys.executable, "scripts/gen_teacher_dataset.py",
       "--manifest", manifest, "--out", str(SMOKE_DIR),
       "--shard-size", "2", "--limit", "4",
       "--device", "cuda:0", "--seed", str(SEED)]
if TEACHER_REVISION:
    cmd += ["--teacher-revision", TEACHER_REVISION]
r = subprocess.run(cmd, cwd=TRAIN_DIR, env=env, capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-2000:])
assert r.returncode == 0, "smoke generation failed (see log above)"

r = subprocess.run([sys.executable, "scripts/validate_teacher_dataset.py",
                    "--root", str(SMOKE_DIR), "--manifest", manifest, "--allow-partial"],
                   cwd=TRAIN_DIR, capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-2000:])
assert r.returncode == 0, "smoke validation failed"

row = torch.load(SMOKE_DIR / "shard_0000.pt", map_location="cpu", weights_only=False)[0]
T, F = len(row["ids"]), row["frames"]
assert row["t_en"].shape == (512, T)
assert row["d"].shape == (T, 640)
assert row["F0"].numel() == row["N"].numel() == 2 * F
assert row["audio"].numel() == 600 * F
assert row["audio"].dtype == torch.int16 and row["pred_dur"].min() >= 1
print(f"smoke row ok: {T} tokens, {F} frames, {row['audio'].numel() / 24000:.2f} s, "
      f"t_en {tuple(row['t_en'].shape)}, d {tuple(row['d'].shape)}")
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

## 4. Generate the full cache

One process per GPU, disjoint shard ranges (`--start 0 --stride 2` and `--start 1
--stride 2`). Each shard is written atomically with a sidecar checksum. Rerunning this cell
skips complete shards, so a dead session resumes where it stopped.

In [ ]:
DATASET_DIR.mkdir(parents=True, exist_ok=True)
shards_before = {p.name for p in DATASET_DIR.glob("shard_*.pt")}
print(f"existing shards: {len(shards_before)} / {n_shards}")

procs = []
for start in range(N_PROCS):
    log_path = WORK / f"gen_start{start}_stride{N_PROCS}.log"
    f = open(log_path, "w")
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(start)
    env["PYTHONUNBUFFERED"] = "1"
    cmd = [sys.executable, "scripts/gen_teacher_dataset.py",
           "--manifest", manifest, "--out", str(DATASET_DIR),
           "--shard-size", str(SHARD_SIZE),
           "--start", str(start), "--stride", str(N_PROCS),
           "--device", "cuda:0", "--seed", str(SEED)]
    if TEACHER_REVISION:
        cmd += ["--teacher-revision", TEACHER_REVISION]
    print(f"[gpu {start}] " + " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=TRAIN_DIR, env=env, stdout=f, stderr=subprocess.STDOUT)
    procs.append({"start": start, "proc": p, "log": log_path, "file": f})

t0 = time.time()
while any(x["proc"].poll() is None for x in procs):
    time.sleep(60)
    free = shutil.disk_usage(DATASET_DIR).free / 1e9
    for x in procs:
        status = "running" if x["proc"].poll() is None else f"exit {x['proc'].returncode}"
        tail = subprocess.run(["tail", "-c", "2500", str(x["log"])],
                              capture_output=True, text=True).stdout
        lines = [ln for ln in tail.replace("\r", "\n").splitlines() if ln.strip()]
        print(f"[gpu {x['start']} {status:9s}] {lines[-1] if lines else ''}", flush=True)
    print(f"[{time.time() - t0:6.0f}s] disk {free:.1f} GB free", flush=True)
    if free < 0.5:
        for x in procs:
            if x["proc"].poll() is None:
                x["proc"].terminate()
        raise RuntimeError("out of disk; see DATASET.md for the /kaggle/temp option")

for x in procs:
    x["file"].close()

bad = [x for x in procs if x["proc"].returncode != 0]
for x in bad:
    print(f"\n===== gpu {x['start']} failed (exit {x['proc'].returncode}) =====")
    lines = open(x["log"], encoding="utf-8", errors="replace").read().replace("\r", "\n").splitlines()
    print("\n".join(lines[-80:]))
assert not bad, "generation failed; rerun this cell to resume"

shards = sorted(DATASET_DIR.glob("shard_*.pt"))
size_gb = sum(p.stat().st_size for p in shards) / 1e9
print(f"done: {len(shards)} shards, {size_gb:.1f} GB, {(time.time() - t0) / 60:.1f} min")
assert len(shards) == n_shards, f"expected {n_shards} shards, found {len(shards)}"

## 5. Validate and write the dataset card

Copies the manifest next to the shards, checks every row and sidecar checksum, and writes
`meta.json`, `splits.json`, `schema.json`, `validate.json`, `README.md`.

In [ ]:
shutil.copy2(TRAIN_DIR / manifest, DATASET_DIR / "hindi_segments.jsonl")
shutil.copy2(TRAIN_DIR / report_path, DATASET_DIR / "hindi_segments_report.json")

cmd = [sys.executable, "scripts/validate_teacher_dataset.py",
       "--root", str(DATASET_DIR),
       "--manifest", str(DATASET_DIR / "hindi_segments.jsonl"),
       "--report", str(DATASET_DIR / "hindi_segments_report.json")]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=TRAIN_DIR, capture_output=True, text=True)
print(r.stdout[-8000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("validation failed; inspect validate.json")

val = json.load(open(DATASET_DIR / "validate.json", encoding="utf-8"))
meta = json.load(open(DATASET_DIR / "meta.json", encoding="utf-8"))
assert val["ok"] and val["complete"], val
assert val["error_count"] == 0
assert meta["counts"]["examples"] == meta["counts"]["expected_examples"] == len(rows)
assert meta["counts"]["missing_examples"] == 0
assert meta["splits"]["counts"] == {k: v for k, v in report["counts"].items() if k != "total"}
print(json.dumps({
    "examples": meta["counts"]["examples"],
    "parents": meta["counts"]["parents"],
    "audio_hours": round(meta["audio"]["hours"], 2),
    "splits": meta["splits"]["counts"],
    "teacher": meta["teacher"],
    "size_gb": round(sum(p.stat().st_size for p in DATASET_DIR.glob('shard_*.pt')) / 1e9, 2),
}, indent=2, ensure_ascii=False))

## 6. Publish the cache

Set `PUBLISH = "kaggle"` above to create or update `KAGGLE_SLUG` as a **private** Kaggle
Dataset. The dataset root is `DATASET_DIR`, so the files mount at
`/kaggle/input/<slug>/shard_*.pt`. If credentials are missing, the cell prints the manual
command and keeps the cache intact.

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def kaggle_creds():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if not os.environ.get(name):
            value = _secret(name)
            if value:
                os.environ[name] = value
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    candidates = glob.glob("/kaggle/input/**/kaggle.json", recursive=True)
    candidates.append(str(Path.home() / ".kaggle" / "kaggle.json"))
    for path in candidates:
        if os.path.isfile(path):
            cfg = json.load(open(path, encoding="utf-8"))
            os.environ.setdefault("KAGGLE_USERNAME", cfg.get("username", ""))
            os.environ.setdefault("KAGGLE_KEY", cfg.get("key", ""))
    return bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))

published = ""
if PUBLISH == "kaggle":
    if not kaggle_creds():
        print("Kaggle credentials not found. Add KAGGLE_USERNAME / KAGGLE_KEY as notebook "
              "secrets, or upload manually:")
        print(f"  kaggle datasets create -p {DATASET_DIR} --dir-mode skip")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                        "kaggle"], check=True)
        kaggle_bin = shutil.which("kaggle") or str(Path(sys.executable).parent / "kaggle")
        if not Path(kaggle_bin).is_file():
            raise RuntimeError("installed the kaggle package but cannot find its CLI")
        dsid = f"{os.environ['KAGGLE_USERNAME']}/{KAGGLE_SLUG}"
        meta_path = DATASET_DIR / "dataset-metadata.json"
        meta_path.write_text(json.dumps({
            "title": "Kokoro Hindi Teacher Cache",
            "id": dsid,
            "licenses": [{"name": "other"}],
        }, indent=2) + "\n", encoding="utf-8")

        print(f"uploading {DATASET_DIR} -> {dsid} (this can take 30-60 min)...", flush=True)
        r = subprocess.run([kaggle_bin, "datasets", "create",
                            "-p", str(DATASET_DIR), "--dir-mode", "skip"],
                           capture_output=True, text=True)
        out = r.stdout + r.stderr
        if r.returncode != 0 and ("already exists" in out.lower() or "409" in out
                                  or "dataset already" in out.lower()):
            r = subprocess.run([kaggle_bin, "datasets", "version",
                                "-p", str(DATASET_DIR),
                                "-m", f"teacher cache {meta['counts']['examples']} rows @ {commit}",
                                "--dir-mode", "skip"],
                               capture_output=True, text=True)
            out = r.stdout + r.stderr
        print(out[-4000:])
        if r.returncode == 0:
            published = f"https://www.kaggle.com/datasets/{dsid}"
            print("published:", published)
        else:
            print("upload failed: the cache is complete and unchanged; rerun this cell.")

elif PUBLISH == "hf":
    if not HF_REPO_ID:
        print("Set HF_REPO_ID (or PUBLISH = 'none').")
    else:
        token = os.environ.get("HF_TOKEN") or _secret("HF_TOKEN")
        if not token:
            print("HF_TOKEN not found; add it as a Kaggle secret or set PUBLISH = 'none'.")
        else:
            from huggingface_hub import HfApi
            api = HfApi(token=token)
            api.create_repo(HF_REPO_ID, repo_type="dataset", exist_ok=True, private=True)
            print(f"uploading {DATASET_DIR} -> {HF_REPO_ID} (this can take 30-60 min)...", flush=True)
            api.upload_folder(folder_path=str(DATASET_DIR), repo_id=HF_REPO_ID,
                              repo_type="dataset",
                              commit_message=f"teacher cache {meta['counts']['examples']} rows @ {commit}")
            published = f"https://huggingface.co/datasets/{HF_REPO_ID}"
            print("published:", published)
else:
    print("PUBLISH = 'none'; nothing uploaded.")

print("\nDataset root:", DATASET_DIR)
print("Files:", len(list(DATASET_DIR.iterdir())))
print("Shards:", len(list(DATASET_DIR.glob("shard_*.pt"))))

## 7. Train from the cache (next notebook)

Create a **new** Kaggle notebook with the published dataset attached, then:

```bash
!git clone --depth 1 https://github.com/Subham460/hinditts.git
%cd hinditts/training
!pip install -q --no-cache-dir kokoro==0.9.4 "misaki[en]==0.9.4"
rm -rf data/teacher && ln -s /kaggle/input/kokoro-hindi-teacher data/teacher
```

```bash
# GPU 0: text side
CUDA_VISIBLE_DEVICES=0 python scripts/train_text.py s+mlp --steps 8000
# GPU 1: decoder (separate process/cell)
CUDA_VISIBLE_DEVICES=1 python scripts/train_decoder.py A --steps 50000
```

The mount is read-only: do **not** run `regen_audio.py` against it. The cache already holds
the teacher waveform and every intermediate. `dataio.py` uses `mmap`, so the shards are
paged from disk instead of filling RAM. The full runbook, schema, and evaluation commands
are in [`training/DATASET.md`](https://github.com/Subham460/hinditts/blob/main/training/DATASET.md).